In [4]:
import pandas as pd

df = pd.read_csv("../data/raw/go_emotions_pl_nllb_final.csv")  # adjust path if needed

def is_corrupted(text: str) -> bool:
    """Check if translation is a repetition loop."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return True
    words = text.split()

    # Too short to be a loop — skip the ratio check
    if len(words) < 8:
        return False

    most_common = max(set(words), key=words.count)
    return words.count(most_common) / len(words) > 0.4  # also raised threshold slightly

corrupted_mask = df["text_pl"].apply(is_corrupted)
print(f"Corrupted rows: {corrupted_mask.sum()} ({corrupted_mask.mean()*100:.1f}%)")
print(corrupted_mask[corrupted_mask].index.tolist()[:20])  # show corrupted indices

Corrupted rows: 44 (0.1%)
[1122, 1463, 1515, 2714, 2733, 3254, 4652, 5165, 9422, 10081, 12712, 13374, 13567, 13664, 14830, 15109, 15432, 15732, 15846, 16049]


In [5]:
corrupted_df = df[corrupted_mask][["text", "text_pl"]]

for idx, row in corrupted_df.iterrows():
    print(f"[{idx}] EN: {row['text']}")
    print(f"      PL: {row['text_pl']}")
    print("---")

[1122] EN: It is. Something I've wondered about many times
      PL: Jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest,
---
[1463] EN: People have made money off it. Belle Delphine was hit with a $60,000 fine recently
      PL: Belle Delphine została niedawno karana 60 tysięcy dolarów karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą kar
---
[1515] EN: What song was that earlier ? It went oooh oooh ooooh oooh oooh oooh.
      PL: To poszło oooh

In [6]:
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Fix 1 reads the PRISTINE original and writes the canonical fixed file used by 01.
SRC_PATH = "../data/raw/go_emotions_pl_nllb_final.csv"     # original, never modified
FIXED_PATH = "../data/raw/go_emotions_pl_nllb_fixed.csv"   # canonical fixed file (01 reads this)
df = pd.read_csv(SRC_PATH)

# Same corruption check
def is_corrupted(text: str) -> bool:
    if not isinstance(text, str) or len(text.strip()) == 0:
        return True
    words = text.split()
    if len(words) < 8:
        return False
    most_common = max(set(words), key=words.count)
    return words.count(most_common) / len(words) > 0.4

corrupted_mask = df["text_pl"].apply(is_corrupted)
print(f"Corrupted rows to fix: {corrupted_mask.sum()}")

# Load 600M model locally
MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"
TGT_LANG = "pol_Latn"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to("cuda")

tgt_lang_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
print(f"Model loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

def translate_single(text: str) -> str:
    tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            forced_bos_token_id=tgt_lang_id,
            max_new_tokens=128,
            num_beams=1,            # greedy — avoids loops
            repetition_penalty=1.5,
            no_repeat_ngram_size=4,
        )

    return tokenizer.decode(output_ids[0].cpu(), skip_special_tokens=True)

# Retranslate only corrupted rows
fixed = 0
for idx in df[corrupted_mask].index:
    original_en = df.loc[idx, "text"]
    new_translation = translate_single(original_en)
    print(f"[{idx}] EN: {original_en}")
    print(f"      OLD: {df.loc[idx, 'text_pl']}")
    print(f"      NEW: {new_translation}")
    print("---")
    df.loc[idx, "text_pl"] = new_translation
    fixed += 1

print(f"\nFixed {fixed} rows.")

# Save corrected file (canonical fixed file used by 01)
df.to_csv(FIXED_PATH, index=False)
print(f"Saved to {FIXED_PATH}")

Corrupted rows to fix: 44
Loading model...


Loading weights: 100%|██████████| 512/512 [00:02<00:00, 250.00it/s] 
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded. VRAM: 5.1 GB


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1122] EN: It is. Something I've wondered about many times
      OLD: Jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest, jest,
      NEW: To coś, o czym zastanawiałem się wiele razy.
---
[1463] EN: People have made money off it. Belle Delphine was hit with a $60,000 fine recently
      OLD: Belle Delphine została niedawno karana 60 tysięcy dolarów karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą karą kar
      NEW: Belle Delphine została ostatnio karana 

...

[41612] EN: [NAME], [NAME], [NAME], [NAME] big 4 >>>
      OLD: [NAME], [NAME], [NAME], [NAME], [NAME] big 4 >>>
      NEW: [NAME], [Name] [NAME], [NAME]. duże 4 >>>
---


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[42147] EN: r/rareinsults also i’m surprised this guy isn’t fussing over the “it’s” instead of “its”
      OLD: r/rareinsults również im zaskoczony ten facet nie jest ss zamiast sss ss ss ss ss ss ss
      NEW: Rareinsults również jestem zaskoczony tym facetem nie jest beszczanie nad jego zamiast jej.
---


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[42692] EN: That sound you are hearing is me, a middle aged woman, applauding you wildly. How fun this is to read on my morning break.
      OLD: To brzmienie, które słyszysz, to ja, kobieta w średnim wieku, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja, ja
      NEW: To brzmi jak ja, kobieta w średnim wieku. Jakże dobrze jest czytać to na poranną przerwie!
---
[43183] EN: Serenity now ! Serenity now !
      OLD: Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now! Serenity now!
      NEW: Serenity , teraz !
---

Fixed 44 rows.
Saved to ../data/raw/go_emotions_pl_nllb_fixed.csv


---
## Fix 2 — truncated multi-sentence translations

NLLB-200 is sentence-level: given several sentences it often emits EOS after the first and
drops the rest. Labels stay but the text no longer justifies them → artificial label noise.
Scale: ~5,800 rows / ~13,000 sentences (13.4%).

**Fix:** rows whose English `text` has ≥2 sentences but whose `text_pl` is <60% of the word
count get each sentence re-translated (in GPU batches) and rejoined.

Run Fix 1 first — Fix 2 reuses its loaded NLLB model.

In [7]:
import re
import pandas as pd

# Work on the canonical fixed file (already loop-fixed by Fix 1).
FIXED_PATH = "../data/raw/go_emotions_pl_nllb_fixed.csv"
df = pd.read_csv(FIXED_PATH)


def count_sentences(text: str) -> int:
    """Rough sentence count: non-empty chunks split on runs of . ! ?"""
    if not isinstance(text, str):
        return 0
    return len([c for c in re.split(r"[.!?]+", text) if c.strip()])


def is_truncated(en: str, pl: str) -> bool:
    """English has >=2 sentences but Polish lost sentences and is much shorter."""
    if not isinstance(en, str) or not isinstance(pl, str) or not pl.strip():
        return False
    if count_sentences(en) < 2:
        return False
    en_w, pl_w = len(en.split()), len(pl.split())
    return count_sentences(pl) < count_sentences(en) and pl_w < 0.6 * en_w


trunc_mask = df.apply(lambda r: is_truncated(r["text"], r["text_pl"]), axis=1)
print(f"Truncated rows: {trunc_mask.sum()} ({trunc_mask.mean() * 100:.1f}%)")
print("(review the count before running the re-translation cell — it is the heavy GPU step)\n")

for idx in df[trunc_mask].index[:15]:
    print(f"[{idx}] EN: {str(df.loc[idx, 'text'])[:130]}")
    print(f"      PL: {str(df.loc[idx, 'text_pl'])[:130]}")
    print("---")

Truncated rows: 5834 (13.4%)
(review the count before running the re-translation cell — it is the heavy GPU step)

[5] EN: OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe PlAyOfFs! Dumbass Broncos fans circa December 2015.
      PL: Omg pEyToN nie jest gotowy, aby pomóc w plAyOfFs!
---
[7] EN: We need more boards and to create a bit more space for [NAME]. Then we’ll be good.
      PL: Potrzebujemy więcej płyt i stworzyć trochę więcej miejsca dla [NAME].
---
[46] EN: It's true though. He either gets no shirt and freezes to death or wears a stupid looking butchers cape. I hope he gets something b
      PL: Albo nie dostaje koszulki i zamraża się na śmierć, albo nosi głupie wyglądające kapsułki rzeźników.
---
[51] EN: Should’ve dumped coke all over her right after the movie, and make a run for it. Fk pettiness 
      PL: Powinienem był po filmie rzucić na nią kokę i uciekać.
---
[57] EN: Lord help me I want to be a mod again. So many damn trolls here
      PL: Boże pomóż mi, chcę znowu być

In [8]:
import torch
import spacy
from collections import defaultdict

# Reuse the NLLB model + tokenizer loaded in Fix 1 (same kernel).
if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("Run the Fix 1 model cell first — it loads the NLLB model/tokenizer.")

# English sentence segmenter — no model download needed (blank pipeline + rule-based sentencizer).
sent_nlp = spacy.blank("en")
sent_nlp.add_pipe("sentencizer")

BATCH_SIZE = 16   # small GPU (~6 GB). Raise to 32/64 if you have VRAM headroom (faster).


def translate_batch(texts: list[str], batch_size: int = BATCH_SIZE) -> list[str]:
    """Translate sentences in padded batches, sorted by length to minimize padding waste.

    Batched generation is ~5-10x faster than one sentence at a time: the GPU stays
    saturated instead of paying per-call launch overhead on every short sentence.
    """
    tokenizer.src_lang = SRC_LANG
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))   # short -> long
    out: list[str] = [""] * len(texts)
    for b in range(0, len(order), batch_size):
        idx_chunk = order[b:b + batch_size]
        chunk = [texts[i] for i in idx_chunk]
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True,
                           max_length=128, padding=True).to("cuda")
        with torch.no_grad():
            ids = model.generate(**inputs, forced_bos_token_id=tgt_lang_id,
                                 max_new_tokens=128, num_beams=1,
                                 repetition_penalty=1.5, no_repeat_ngram_size=4)
        for j, dec in zip(idx_chunk, tokenizer.batch_decode(ids, skip_special_tokens=True)):
            out[j] = dec
        if (b // batch_size) % 25 == 0:
            print(f"    {min(b + batch_size, len(order))}/{len(order)} sentences", flush=True)
    return out


# Flatten sentences across all truncated rows, remembering which row each belongs to.
row_idx = list(df[trunc_mask].index)
flat, owner = [], []
for idx in row_idx:
    sents = [s.text.strip() for s in sent_nlp(str(df.loc[idx, "text"])).sents if s.text.strip()]
    if not sents:
        sents = [str(df.loc[idx, "text"])]
    flat.extend(sents)
    owner.extend([idx] * len(sents))

print(f"Translating {len(flat)} sentences from {len(row_idx)} rows (batch={BATCH_SIZE})...")
translated = translate_batch(flat)

# Reassemble per row and write back.
parts = defaultdict(list)
for o, t in zip(owner, translated):
    parts[o].append(t)
for idx in row_idx:
    df.loc[idx, "text_pl"] = " ".join(parts[idx])

df.to_csv(FIXED_PATH, index=False)
print(f"\nRe-translated {len(row_idx)} rows. Saved both fixes back to {FIXED_PATH}")
print("Re-run 01_data_preparation.ipynb to propagate into data/processed/.")

Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Translating 13192 sentences from 5834 rows (batch=16)...
    16/13192 sentences


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    416/13192 sentences


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

    12416/13192 sentences


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    12816/13192 sentences


Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Re-translated 5834 rows. Saved both fixes back to ../data/raw/go_emotions_pl_nllb_fixed.csv
Re-run 01_data_preparation.ipynb to propagate into data/processed/.
